# Grade annotations

Load one annotator's returned `answers_<name>.csv`, decode it with `key.xlsx`,
and score it: an answer is **correct** when the idea behind the chosen A/B slot
is the gold winner. `unclear` is never correct — it is counted separately.

Reading of the groups: `key.xlsx`'s `group` column says which groups a batch has.
Where a batch mixes them, `controls` should be near-perfect (the judge got these
right by a 6-vs-0 margin), `mistakes` are the cases under study, and `hard` are
pairs a strong judge got confidently wrong. A `*-mistakes` batch is nothing but
cases the judge got wrong, so there `agrees_judge` near 0 is the expected reading,
not a red flag.

Drop each returned `answers_<name>.csv` into the batch's `answers/` folder before
running — a Jupyter kernel is not allowed to read `~/Downloads` on macOS, so
pointing at the file where the browser left it fails with a `PermissionError`.

In [ ]:
import json
import re
import textwrap
from pathlib import Path

import pandas as pd

BATCH_DIR = Path("../../../output/judge_error_annotations/batch_opus_4-6_mistakes")
ANNOTATOR = "tom"
ANSWERS = BATCH_DIR / "answers" / f"answers_{ANNOTATOR}.csv"

# BATCH_DIR and ANNOTATOR have to be changed together: every batch has its own
# annotators, and pointing at one from another batch only fails at read_csv.
if not ANSWERS.exists():
    back = sorted(f.name[len("answers_"):-len(".csv")] for f in (BATCH_DIR / "answers").glob("answers_*.csv"))
    raise FileNotFoundError(
        f"{BATCH_DIR.name} has no answers for {ANNOTATOR!r}. "
        f"Returned so far: {', '.join(back) or 'nobody'}."
    )

answers = pd.read_csv(ANSWERS, dtype=str).fillna("")
key = pd.read_excel(BATCH_DIR / "key.xlsx", dtype=str)
key = key[key["annotator"] == ANNOTATOR]

df = key.merge(answers, on="item_id", how="left", validate="one_to_one")
df[["your_choice", "notes"]] = df[["your_choice", "notes"]].fillna("")
print(f"{len(df)} items, {(df['your_choice'] != '').sum()} answered")

## Decode and score

In [ ]:
def chosen_idea(row):
    """Turn the A/B slot the annotator clicked back into an idea id."""
    return {"A": row["shown_A_idea"], "B": row["shown_B_idea"]}.get(row["your_choice"], "unclear")


df["human_idea"] = df.apply(chosen_idea, axis=1)
df["gold_ab"] = ["A" if a == gold else "B" for a, gold in zip(df["shown_A_idea"], df["gold_idea"])]
df["judge_ab"] = ["A" if a == j else "B" if j in ("0", "1") else j
                  for a, j in zip(df["shown_A_idea"], df["judge_novelty_winner"])]
df["is_unclear"] = df["human_idea"] == "unclear"
df["correct"] = df["human_idea"] == df["gold_idea"]
df["agrees_judge"] = df["human_idea"] == df["judge_novelty_winner"]

df[["item_id", "group", "your_choice", "human_idea", "gold_idea", "gold_ab",
    "judge_novelty_winner", "correct", "agrees_judge", "notes"]]

## Summary

`accuracy` counts `unclear` as wrong; `accuracy_decisive` drops those rows.

In [ ]:
def summarize(g):
    decisive = g[~g["is_unclear"]]
    return pd.Series({
        "n": len(g),
        "unclear": int(g["is_unclear"].sum()),
        "correct": int(g["correct"].sum()),
        "wrong": int((~g["correct"] & ~g["is_unclear"]).sum()),
        "accuracy": g["correct"].mean(),
        "accuracy_decisive": decisive["correct"].mean() if len(decisive) else float("nan"),
        "agrees_judge": decisive["agrees_judge"].mean() if len(decisive) else float("nan"),
    })


summary = df.groupby("group").apply(summarize, include_groups=False)
summary.loc["ALL"] = summarize(df)
summary.round(3)

## Notes the annotator left

In [ ]:
for _, row in df[df["notes"].str.strip().ne("")].iterrows():
    print(f"{row['item_id']}  {row['group']:<9} {row['your_choice']:<8} gold: {row['gold_ab']}")
    print(textwrap.fill(row["notes"], width=90, initial_indent="    ", subsequent_indent="    "))
    print()

## Read the pairs

The idea texts come from the annotator's own HTML page, so the A/B slots are exactly
the ones they saw. Only the items the annotator left a note on — those are the ones worth reading.
Widen `SHOW` for more: `df[~df["correct"]]`, `df[df["group"] == "mistakes"]`, or `df`.

In [ ]:
page = (BATCH_DIR / f"{ANNOTATOR}.html").read_text(encoding="utf-8")
items = json.loads(re.search(r'<script[^>]*id="items"[^>]*>(.*?)</script>', page, re.S).group(1))
texts = {it["item_id"]: it for it in items}

SHOW = df[df["notes"].str.strip().ne("")]

for _, row in SHOW.iterrows():
    it = texts[row["item_id"]]
    mark = "correct" if row["correct"] else "unclear" if row["is_unclear"] else "wrong"
    print("=" * 92)
    print(f"{row['item_id']}  {row['group']:<9} {it['area']}")
    print(f"chose: {row['your_choice']:<8} gold: {row['gold_ab']}   judge: {row['judge_ab']:<4} -> {mark}")
    for slot in ("A", "B"):
        tags = [t for t, on in (("gold", row["gold_ab"] == slot),
                                ("judge", row["judge_ab"] == slot),
                                ("chosen", row["your_choice"] == slot)) if on]
        print(f"\n  IDEA {slot}" + (f"  [{', '.join(tags)}]" if tags else ""))
        print(textwrap.fill(it[f"{slot}_text"], width=88,
                            initial_indent="    ", subsequent_indent="    "))
    if row["notes"].strip():
        print("\n  NOTES")
        print(textwrap.fill(row["notes"], width=88,
                            initial_indent="    ", subsequent_indent="    "))
    print()